In [ ]:
import numpy as np
import json
import matplotlib
import matplotlib.pyplot as plt
import os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter, MultipleLocator


In [ ]:
locations_path = './train_v3_tf_logs.json'
if locations_path is None:
    raise FileNotFoundError('Cannot find vis_data_locations.json from current working directory')
with open(locations_path, 'r') as f:
    log_dir_locations = json.load(f)
for key, value in log_dir_locations.items():
    globals()[key] = value

def extract_tensorboard_data(log_dir, run_name_to_tag, step_start=0, step_end=None):
	# to find out event file
	event_file = None
	for file in os.listdir(log_dir):
		if os.path.isfile(os.path.join(log_dir, file)) and file.startswith('events'):
			event_file = os.path.join(log_dir, file)
			break
	runs_data = {}
	
	# Iterate over each run directory
	for run_name in os.listdir(log_dir):
		# Check if the run is in the list of runs to consider
		if run_name not in run_name_to_tag:
			continue
		if run_name_to_tag[run_name] is None:
			continue
		run_path = os.path.join(log_dir, run_name)
		
		# Load event data from the run directory
		event_acc = EventAccumulator(run_path)
		event_acc.Reload()
		
		# Check if the tag is available in the run
		tag = run_name_to_tag[run_name]
		if tag in event_acc.Tags()['scalars']:
			steps = []
			values = []
			for scalar_event in event_acc.Scalars(tag):
				if scalar_event.step < step_start:
					continue
				if step_end is not None and scalar_event.step > step_end:
					break
				steps.append(scalar_event.step)
				values.append(scalar_event.value)
				
			runs_data[run_name] = {'steps': steps, 'values': values}
		else:
			raise ValueError(f'Tag {tag} not found in run {run_name}')
	
	if event_file is not None:
		event_acc = EventAccumulator(event_file)

		# Load all events from the file
		event_acc.Reload()

		for tag in run_name_to_tag:
			if run_name_to_tag[tag] is None:
				if tag in event_acc.Tags()['scalars']:
					steps = []
					values = []
					for scalar_event in event_acc.Scalars(tag):
						if scalar_event.step < step_start:
							continue
						if step_end is not None and scalar_event.step > step_end:
							break
						steps.append(scalar_event.step)
						values.append(scalar_event.value)
					
					runs_data[tag] = {'steps': steps, 'values': values}

	return runs_data

run_name_to_tag = {
	'eval_metrics_average_speed_all': 'eval_metrics/average_speed',
	'eval_metrics_average_speed_all:dummy': 'eval_metrics/average_speed',
	'eval_metrics_co2_emission_all': 'eval_metrics/co2_emission',
	'eval_metrics_co2_emission_all:dummy': 'eval_metrics/co2_emission',
	'eval_metrics_delay_all': 'eval_metrics/delay',
	'eval_metrics_delay_all:dummy': 'eval_metrics/delay',
	'eval_metrics_flow_all': 'eval_metrics/flow',
	'eval_metrics_flow_all:dummy': 'eval_metrics/flow',
	'eval_metrics_lanechange_count_all': 'eval_metrics/lanechange_count',
	'eval_metrics_lanechange_count_all:dummy': 'eval_metrics/lanechange_count',
	'eval_metrics_total_time_all': 'eval_metrics/total_time',
	'eval_metrics_total_time_all:dummy': 'eval_metrics/total_time',
	'eval_metrics_travel_time_all': 'eval_metrics/travel_time',
	'eval_metrics_travel_time_all:dummy': 'eval_metrics/travel_time',
	'eval_metrics_lanechange_count_all': 'eval_metrics/lanechange_count',
	'eval_metrics_lanechange_count_all:dummy': 'eval_metrics/lanechange_count',
	'training_episode_agent_reward_avg': 'training_episode_agent_reward',
	'training_episode_agent_reward_50pt': 'training_episode_agent_reward',
	'training_episode_agent_reward_90pt': 'training_episode_agent_reward',
	'training_episode_reward_avg': 'training_episode_reward',
	'training_episode_reward_50pt': 'training_episode_reward',
	'training_episode_reward_90pt': 'training_episode_reward',
	'training_episode_reward_spi_avg': 'training_episode_reward_spi',
	'training_episode_reward_los_avg': 'training_episode_reward_los',
	'training_episode_global_reward_avg': 'training_episode_global_reward',
	'training_episode_global_reward_50pt': 'training_episode_global_reward',
	'training_episode_global_reward_90pt': 'training_episode_global_reward',
	'q_values': None,
	'episode_metrics_lanechange_count_all': 'episode_metrics/lanechange_count',
	'episode_length': 'episode_length',
}


In [ ]:
data_dict = {}
data_dict["dummy_010_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_dqn, run_name_to_tag)
data_dict["dummy_015_data"] = extract_tensorboard_data(tf_log_dir_015_dummy_dqn, run_name_to_tag)
data_dict["dummy_030_data"] = extract_tensorboard_data(tf_log_dir_030_dummy_dqn, run_name_to_tag)
# data_dict["dummy_045_data"] = extract_tensorboard_data(tf_log_dir_045_dummy_dqn, run_name_to_tag)

data_dict["ld_010_data"] = extract_tensorboard_data(tf_log_dir_010_lane_degrade_dqn, run_name_to_tag)
data_dict["ld_015_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_dqn, run_name_to_tag)
data_dict["ld_030_data"] = extract_tensorboard_data(tf_log_dir_030_lane_degrade_dqn, run_name_to_tag)
# data_dict["ld_045_data"] = extract_tensorboard_data(tf_log_dir_045_lane_degrade_dqn, run_name_to_tag)

data_dict["vs_010_data"] = extract_tensorboard_data(tf_log_dir_010_vehicle_stop_dqn, run_name_to_tag)
data_dict["vs_015_data"] = extract_tensorboard_data(tf_log_dir_015_vehicle_stop_dqn, run_name_to_tag)
data_dict["vs_030_data"] = extract_tensorboard_data(tf_log_dir_030_vehicle_stop_dqn, run_name_to_tag)
# data_dict["vs_045_data"] = extract_tensorboard_data(tf_log_dir_045_vehicle_stop_dqn, run_name_to_tag)

# data_dict["dummy_010_cr90_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr90_dqn, run_name_to_tag)
# data_dict["dummy_010_cr80_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr80_dqn, run_name_to_tag)
# data_dict["dummy_010_cr70_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr70_dqn, run_name_to_tag)
# data_dict["dummy_010_cr60_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr60_dqn, run_name_to_tag)
# data_dict["dummy_010_cr50_data"] = extract_tensorboard_data(tf_log_dir_010_dummy_cr50_dqn, run_name_to_tag)

# data_dict["ld_015_cr90_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_cr90_dqn, run_name_to_tag)
# data_dict["ld_015_cr80_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_cr80_dqn, run_name_to_tag)
# data_dict["ld_015_cr70_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_cr70_dqn, run_name_to_tag)
# data_dict["ld_015_cr60_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_cr60_dqn, run_name_to_tag)
# data_dict["ld_015_cr50_data"] = extract_tensorboard_data(tf_log_dir_015_lane_degrade_cr50_dqn, run_name_to_tag)


In [ ]:
# data in the format of {run_name: {'steps': [step1, step2, ...], 'values': [value1, value2, ...]}},
# plot the data of eval_metrics_average_speed_all and eval_metrics_average_speed_all:dummy in the same plot
# use color 'blue' for the data of eval_metrics_average_speed_all and color 'red' for the data of eval_metrics_average_speed_all:dummy
# add a legend to the plot
# plot moving average of the data with window size 10
# plot the real data with alpha=0.5 and the moving average with alpha=1
# add a title to the plot

def moving_average(data, window_size):
	# output length equal input length
	ma_data = np.zeros_like(data)
	data_filtered = np.zeros_like(data)
	data_filtered[0] = data[0]
	for i in range(1, len(data)):
		if data[i] > 0.:
			data_filtered[i] = data[i]
		else:
			data_filtered[i] = data_filtered[i-1]

	for i in range(len(data)):
		if i < window_size-1:
			ma_data[i] = np.mean(data_filtered[:i+1])
		else:
			ma_data[i] = np.mean(data_filtered[i-window_size+1:i+1])
	return ma_data

def exponential_moving_average(data, alpha):
	ema_data = np.zeros_like(data)
	ema_data[0] = data[0]
	for i in range(1, len(data)):
		if data[i] > 0.:
			ema_data[i] = alpha * data[i] + (1 - alpha) * ema_data[i-1]
		else:
			ema_data[i] = ema_data[i-1]
	return ema_data

def plot_data(data, keys, labels, colors, title=None, save_pth=None, window_size=10, kargs={}):
	save_folder = os.path.dirname(save_pth) if save_pth else None
	if save_folder and not os.path.exists(save_folder):
		os.makedirs(save_folder)
	font = {
		'family' : 'Times New Roman',
		'size': 8
		}
	if "font" in kargs:
		font.update(kargs["font"])
	matplotlib.rc('font', **font)
	matplotlib.rcParams['axes.linewidth'] = 0.5
	figsize = kargs.get("figsize", (6, 6))
	fig, ax = plt.subplots(figsize=figsize)
	min_epi_length = kargs.get("min_epi_length", None)
	print(f"min_epi_length: {min_epi_length}")
	print(f"data keys: {list(data.keys())}")
	if min_epi_length is not None and 'episode_length' in data:
		epi_length_data = data['episode_length']['values']
		valid_indices = [i for i, length in enumerate(epi_length_data) if length >= min_epi_length]
		for key in keys:
			data[key]['steps'] = [data[key]['steps'][i] for i in valid_indices]
			data[key]['values'] = [data[key]['values'][i] for i in valid_indices]
	for key, label, color in zip(keys, labels, colors):
		steps = data[key]['steps']
		values = data[key]['values']
		ma_values = moving_average(values, window_size)
		if kargs.get("ema_alpha") is not None:
			ma_values = exponential_moving_average(values, kargs["ema_alpha"])
		if label:
			plt.plot(steps, ma_values, color=color, alpha=1, label=label, linewidth=1)
		else:
			plt.plot(steps, ma_values, color=color, alpha=1, linewidth=1)
		if kargs.get("alpha", 0.2) > 0:
			plt.plot(steps, values, color=color, alpha=kargs.get("alpha", 0.2), linewidth=0.6)
	plt.xlabel(kargs.get("xlabel", "Learning steps"))
	plt.ylabel(kargs.get("ylabel", "Average speed (m/s)"))
	# Function to format xticks
	def format_xticks(value, tick_number):
		return f"{int(value / 1000)}k"
	if title:
		plt.title(title)
	if kargs.get("y_space"):
		ax.yaxis.set_major_locator(ticker.MultipleLocator(kargs.get("y_space")))
	if kargs.get("xlim"):
		plt.xlim(kargs["xlim"])
	if kargs.get("ylim"):
		plt.ylim(kargs["ylim"])
	# Add this inside your plot_data function
	ax.xaxis.set_major_locator(MultipleLocator(50000))  # Interval of 50k
	ax.xaxis.set_major_formatter(FuncFormatter(format_xticks))
	plt.grid(axis = 'y', linestyle='dotted', linewidth=0.5)
	if kargs.get("legend", True):
		legend_fontsize = kargs.get("legend_fontsize", 8)  # Default to 8 if not specified
		plt.legend(fontsize=legend_fontsize)
	# plt.tight_layout()
	if save_pth:
		plt.savefig(save_pth, dpi=300, bbox_inches = "tight")
	# plt.title('Average speed')
	plt.show()

def summary_lane_change_count(data_dict):
	key = 'eval_metrics_lanechange_count_all'
	key_baseline = 'eval_metrics_lanechange_count_all:dummy'
	keys = [key, key_baseline]
	for label, data in data_dict.items():
		num_episode = len(data[key]['values'])
		avg_lc = np.mean(data[key]['values'])
		stdv_lc = np.std(data[key]['values'])
		avg_lc_baseline = np.mean(data[key_baseline]['values'])
		stdv_lc_baseline = np.std(data[key_baseline]['values'])
		print(f'{label}: number of episode:{num_episode}, {avg_lc:.3f}$\pm${stdv_lc:.3f} vs {avg_lc_baseline:.3f}$\pm${stdv_lc_baseline:.3f}')
	return

In [ ]:
lc_data_dict = {
	'dummy_rho_010': data_dict["dummy_010_data"],
	'dummy_rho_015': data_dict["dummy_015_data"],
	'dummy_rho_030': data_dict["dummy_030_data"],
	# 'dummy_rho_045': data_dict["dummy_045_data"],

}
summary_lane_change_count(lc_data_dict)

lc_data_dict = {
	'lane_degrade_010': data_dict["ld_010_data"],
	'lane_degrade_015': data_dict["ld_015_data"],
	'lane_degrade_030': data_dict["ld_030_data"],
	# 'lane_degrade_045': data_dict["ld_045_data"],
}
summary_lane_change_count(lc_data_dict)

lc_data_dict = {
	'vehicle_stop_010': data_dict["vs_010_data"],
	'vehicle_stop_015': data_dict["vs_015_data"],
	'vehicle_stop_030': data_dict["vs_030_data"],
	# 'vehicle_stop_045': data_dict["vs_045_data"],
}
summary_lane_change_count(lc_data_dict)

In [ ]:
plot_data_dict = {
    '010_dummy_reward': data_dict["dummy_010_data"]['training_episode_reward_avg'],
    '010_ld_reward': data_dict["ld_010_data"]['training_episode_reward_avg'],
    '010_vs_reward': data_dict["vs_010_data"]['training_episode_reward_avg']
}
keys = ['010_dummy_reward', '010_ld_reward', '010_vs_reward']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = 'Low Demand'
title = None
colors = ['blue', 'red', 'green']
ylim = (16.4, 17.2)
kargs = {"xlim": (2000, 150000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2, "ema_alpha": 0.02, "min_epi_length": 300}
save_path = './train_vis_new/010_reward.pdf'
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)


plot_data_dict = {
    '015_dummy_reward': data_dict["dummy_015_data"]['training_episode_reward_avg'],
	'015_ld_reward': data_dict["ld_015_data"]['training_episode_reward_avg'],
	'015_vs_reward': data_dict["vs_015_data"]['training_episode_reward_avg']
}
keys = ['015_dummy_reward', '015_ld_reward', '015_vs_reward']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (16.0, 16.8)
kargs = {"xlim": (2000, 150000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2, "ema_alpha": 0.02, "min_epi_length": 300}
save_path = './train_vis_new/015_reward.pdf'
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

plot_data_dict = {
    '030_dummy_reward': data_dict["dummy_030_data"]['training_episode_reward_avg'],
    '030_ld_reward': data_dict["ld_030_data"]['training_episode_reward_avg'],
    '030_vs_reward': data_dict["vs_030_data"]['training_episode_reward_avg']
}
keys = ['030_dummy_reward', '030_ld_reward', '030_vs_reward']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (12.0, 13.4)
kargs = {"xlim": (2000, 150000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2, "ema_alpha": 0.02, "min_epi_length": 300}
save_path = './train_vis_new/030_reward.pdf'
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

# plot_data_dict = {
#     '045_dummy_reward': data_dict["dummy_045_data"]['training_episode_reward_avg'],
# 	'045_ld_reward': data_dict["ld_045_data"]['training_episode_reward_avg'],
# 	'045_vs_reward': data_dict["vs_045_data"]['training_episode_reward_avg']
# }
# keys = ['045_dummy_reward', '045_ld_reward', '045_vs_reward']
# labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
# title = None
# colors = ['blue', 'red', 'green']
# ylim = (9.6, 10.6)
# kargs = {"xlim": (2000, 150000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
# "font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2, "ema_alpha": 0.1}
# save_path = './train_vis_new/045_reward.pdf'
# plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

In [ ]:
plot_data_dict = {
    '010_dummy_reward_cr100': data_dict["dummy_010_data"]['training_episode_reward_avg'],
	'010_dummy_reward_cr90': data_dict["dummy_010_cr90_data"]['training_episode_reward_avg'],
	'010_dummy_reward_cr80': data_dict["dummy_010_cr80_data"]['training_episode_reward_avg'],
	'010_dummy_reward_cr70': data_dict["dummy_010_cr70_data"]['training_episode_reward_avg'],
	'010_dummy_reward_cr60': data_dict["dummy_010_cr60_data"]['training_episode_reward_avg'],
	'010_dummy_reward_cr50': data_dict["dummy_010_cr50_data"]['training_episode_reward_avg']
}
keys = ['010_dummy_reward_cr100', '010_dummy_reward_cr90', '010_dummy_reward_cr80', '010_dummy_reward_cr70', '010_dummy_reward_cr60', '010_dummy_reward_cr50']
labels = ['100%', '90%', '80%', '70%', '60%', '50%']
title = None
colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown']
ylim = (16.6, 17.0)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": -0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 10}, "ylim": ylim, "y_space": 0.1, "legend_fontsize": 6}
save_path = './train_vis_new/010_dummy_control_rate_reward.pdf'
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)



plot_data_dict = {
    '015_ld_reward_cr100': data_dict["ld_015_data"]['training_episode_reward_avg'],
	'015_ld_reward_cr90': data_dict["ld_015_cr90_data"]['training_episode_reward_avg'],
	'015_ld_reward_cr80': data_dict["ld_015_cr80_data"]['training_episode_reward_avg'],
	'015_ld_reward_cr70': data_dict["ld_015_cr70_data"]['training_episode_reward_avg'],
	'015_ld_reward_cr60': data_dict["ld_015_cr60_data"]['training_episode_reward_avg'],
	'015_ld_reward_cr50': data_dict["ld_015_cr50_data"]['training_episode_reward_avg']
}

keys = ['015_ld_reward_cr100', '015_ld_reward_cr90', '015_ld_reward_cr80', '015_ld_reward_cr70', '015_ld_reward_cr60', '015_ld_reward_cr50']
labels = ['100%', '90%', '80%', '70%', '60%', '50%']
title = None
colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown']
ylim = (16.3, 16.7)
# ylim = None
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": -0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 10}, "ylim": ylim, "y_space": 0.1, "legend_fontsize": 6}
save_path = './train_vis_new/015_ld_control_rate_reward.pdf'
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

In [ ]:
plot_data_dict = {
    'dummy': data_dict["dummy_010_data"]['training_episode_reward_spi_avg'],
	'ld': data_dict["ld_010_data"]['training_episode_reward_spi_avg'],
	'vs': data_dict["vs_010_data"]['training_episode_reward_spi_avg']
}
keys = ['dummy', 'ld', 'vs']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (15.0, 16.0)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2}
save_path = './train_vis_new/010_spi_reward.pdf'
# save_path = None
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

plot_data_dict = {
    'dummy': data_dict["dummy_010_data"]['training_episode_reward_los_avg'],
	'ld': data_dict["ld_010_data"]['training_episode_reward_los_avg'],
	'vs': data_dict["vs_010_data"]['training_episode_reward_los_avg']
}
keys = ['dummy', 'ld', 'vs']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (17.6, 18.6)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2}
save_path = './train_vis_new/010_los_reward.pdf'
# save_path = None
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)


plot_data_dict = {
    'dummy': data_dict["dummy_015_data"]['training_episode_reward_spi_avg'],
	'ld': data_dict["ld_015_data"]['training_episode_reward_spi_avg'],
	'vs': data_dict["vs_015_data"]['training_episode_reward_spi_avg']
}
keys = ['dummy', 'ld', 'vs']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (15.4, 16.4)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2}
save_path = './train_vis_new/015_spi_reward.pdf'
# save_path = None
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

plot_data_dict = {
    'dummy': data_dict["dummy_015_data"]['training_episode_reward_los_avg'],
	'ld': data_dict["ld_015_data"]['training_episode_reward_los_avg'],
	'vs': data_dict["vs_015_data"]['training_episode_reward_los_avg']
}
keys = ['dummy', 'ld', 'vs']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (16.6, 17.6)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2}
save_path = './train_vis_new/015_los_reward.pdf'
# save_path = None
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)


plot_data_dict = {
    'dummy': data_dict["dummy_045_data"]['training_episode_reward_spi_avg'],
	'ld': data_dict["ld_045_data"]['training_episode_reward_spi_avg'],
	'vs': data_dict["vs_045_data"]['training_episode_reward_spi_avg']
}
keys = ['dummy', 'ld', 'vs']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (6.8, 7.8)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2}
save_path = './train_vis_new/045_spi_reward.pdf'
# save_path = None
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)

plot_data_dict = {
    'dummy': data_dict["dummy_045_data"]['training_episode_reward_los_avg'],
	'ld': data_dict["ld_045_data"]['training_episode_reward_los_avg'],
	'vs': data_dict["vs_045_data"]['training_episode_reward_los_avg']
}
keys = ['dummy', 'ld', 'vs']
labels = ['Stable Flow', 'Lane Degrade', 'Vehicle Stop']
title = None
colors = ['blue', 'red', 'green']
ylim = (12.4, 13.4)
kargs = {"xlim": (2000, 250000), "xlabel": "Learning steps", "ylabel": "Agent Avg Reward", "alpha": 0.03, "figsize": (2.5, 2.5),
"font": {'family' : 'Times New Roman','size': 9}, "ylim": ylim, "y_space": 0.2}
save_path = './train_vis_new/045_los_reward.pdf'
# save_path = None
plot_data(plot_data_dict, keys, labels, colors, title=title, save_pth=save_path, window_size=60, kargs=kargs)